In [1]:
import feedparser
import pandas as pd
from datetime import date, timedelta
from sqlalchemy import create_engine

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

clean_path = "../data/clean/"
raw_path = "../data/raw/"
csv_path = "\\Users\\User\\iCloudDrive\\"
one_path = "\\Users\\User\\OneDrive\\Documents\\Data\\"
pdf_path = "../PDF/"

pd.set_option('display.max_colwidth', 255)
pd.set_option('display.max_rows',None)
url = "https://feeds.feedburner.com/Setorth-form45-en"

today = date.today()
year = 2022
mmdd_str = today.strftime('%m%d')
mmdd_str

'0211'

In [2]:
today = date(2023, 2, 10)
mmdd_str = today.strftime('%m%d')
mmdd_str

'0210'

In [3]:
rss_source = feedparser.parse(url)
f45_number = len(rss_source.entries)
print("Number of F45: ", f45_number)

Number of F45:  23


In [4]:
f45_items = []

for x in range(f45_number):
    f45_content = rss_source.entries[x]
    f45_item = {}
    
    print("\n----------------------------------\n")
    
    print("F45: " + str(x))
    title_ary = f45_content.title.partition(' ')
    f45_item['name'] = title_ary[0].strip() 
    print("Title: ", f45_item['name'])  
    f45_item['year'] = year
    print("Year: ", f45_item['year'])      
    qtr_ary = title_ary[2].partition(' (F45)')
    f45_item['quarter'] = qtr_ary[0][-1]    
    print("Quarter: ", f45_item['quarter'])    
    f45_item['link'] = f45_content.link
    print("Link: ", f45_item['link'])
    f45_item['published'] = f45_content.published
    print("Published: ", f45_item['published'])  
    f45_items.append(f45_item)


----------------------------------

F45: 0
Title:  QTC
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861843460&symbol=QTC
Published:  Fri, 10 Feb 2023 21:53:36 +0700

----------------------------------

F45: 1
Title:  THCOM
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861841810&symbol=THCOM
Published:  Fri, 10 Feb 2023 20:18:48 +0700

----------------------------------

F45: 2
Title:  KTC
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861840250&symbol=KTC
Published:  Fri, 10 Feb 2023 19:33:35 +0700

----------------------------------

F45: 3
Title:  GGC
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861839290&symbol=GGC
Published:  Fri, 10 Feb 2023 19:07:36 +0700

----------------------------------

F45: 4
Title:  SCCC
Year:  2022
Quarter:  y
Link:  https://www.set.or.th/en/market/

In [5]:
df = pd.DataFrame(f45_items)
df[['name','year','quarter','published']]

,name,year,quarter,published
0,QTC,2022,y,"Fri, 10 Feb 2023 21:53:36 +0700"
1,THCOM,2022,y,"Fri, 10 Feb 2023 20:18:48 +0700"
2,KTC,2022,y,"Fri, 10 Feb 2023 19:33:35 +0700"
3,GGC,2022,y,"Fri, 10 Feb 2023 19:07:36 +0700"
4,SCCC,2022,y,"Fri, 10 Feb 2023 18:56:28 +0700"
5,TPIPP,2022,y,"Fri, 10 Feb 2023 18:39:46 +0700"
6,GPSC,2022,y,"Fri, 10 Feb 2023 18:35:41 +0700"
7,NOK,2022,2,"Fri, 10 Feb 2023 18:18:08 +0700"
8,KTIS,2022,1,"Fri, 10 Feb 2023 18:11:32 +0700"
9,TDEX,2022,e,"Fri, 10 Feb 2023 18:02:48 +0700"


In [6]:
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           4     4          4
     2           3     3          3
     3           2     2          2
     e           1     1          1
     y          13    13         13

In [7]:
df.loc[(df.quarter == 'y') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           4     4          4
     2           3     3          3
     3           2     2          2
     e           1     1          1
     4          13    13         13

In [8]:
df.loc[(df.quarter == 'e') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           4     4          4
     2           3     3          3
     3           2     2          2
     4          14    14         14

In [10]:
df.loc[(df.quarter == '2') ,['year','quarter']] = ['2022','4']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     3           1     1          1
     4           6     6          6

In [10]:
df.loc[(df.name == 'OISHI') ,['year','quarter']] = ['2023','1']
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           1     1          1
     2           1     1          1
     3           1     1          1
     4           1     1          1
2023 1           2     2          2

### After change all illogical quarters

In [9]:
df.quarter = df.quarter.astype(int)
df.groupby(['year','quarter']).count()

name  link  published
year quarter                       
2022 1           4     4          4
     2           3     3          3
     3           2     2          2
     4          14    14         14

In [10]:
### First equals to latest published pdf file
df = df.drop_duplicates(subset='name', keep='first')
df.shape

(23, 5)

In [11]:
file_name = 'F45-RAW-' + mmdd_str + '.csv'
raw_file = raw_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name

df[['name','year','quarter','published']].to_csv(output_file, header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(one_file,    header=True, index=False, sep=',')
df[['name','year','quarter','published']].to_csv(raw_file,    header=True, index=False, sep=',')

In [12]:
sql = '''
SELECT *
FROM exempts
ORDER BY name'''
df_exempts = pd.read_sql(sql, conlt)
df_exempts.shape[0]

403

In [13]:
df_merge = pd.merge(df, df_exempts, on='name', how='outer', indicator=True)
df_merge.shape

(421, 7)

### Tickers that are in Exempts table

In [14]:
in_exempts = df_merge.loc[
    df_merge['_merge'] == 'both',
    ['name','year','quarter','published','link']
    
]
in_exempts.year = in_exempts.year.astype(int)
in_exempts.quarter = in_exempts.quarter.astype(int)
in_exempts.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
0,QTC,2022,4,"Fri, 10 Feb 2023 21:53:36 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861843460&symbol=QTC
1,THCOM,2022,4,"Fri, 10 Feb 2023 20:18:48 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861841810&symbol=THCOM
8,KTIS,2022,1,"Fri, 10 Feb 2023 18:11:32 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861833860&symbol=KTIS
14,NINE,2022,4,"Fri, 10 Feb 2023 17:26:54 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861829670&symbol=NINE
18,UV,2022,1,"Fri, 10 Feb 2023 17:06:56 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861827270&symbol=UV


In [15]:
in_exempts.sort_values(by=['published'],ascending=[False]).shape[0]

5

### Not in exempts table

In [16]:
df_out = df_merge.loc[
    df_merge['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]
df_out.year = df_out.year.astype(int)
df_out.quarter = df_out.quarter.astype(int)
df_out.sort_values(by=['published'],ascending=[False])

,name,year,quarter,published,link
2,KTC,2022,4,"Fri, 10 Feb 2023 19:33:35 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861840250&symbol=KTC
3,GGC,2022,4,"Fri, 10 Feb 2023 19:07:36 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861839290&symbol=GGC
4,SCCC,2022,4,"Fri, 10 Feb 2023 18:56:28 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861838420&symbol=SCCC
5,TPIPP,2022,4,"Fri, 10 Feb 2023 18:39:46 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836440&symbol=TPIPP
6,GPSC,2022,4,"Fri, 10 Feb 2023 18:35:41 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836180&symbol=GPSC
7,NOK,2022,2,"Fri, 10 Feb 2023 18:18:08 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861834510&symbol=NOK
9,TDEX,2022,4,"Fri, 10 Feb 2023 18:02:48 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832900&symbol=TDEX
10,METCO,2022,1,"Fri, 10 Feb 2023 17:53:07 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832470&symbol=METCO
11,INTUCH,2022,4,"Fri, 10 Feb 2023 17:53:03 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832440&symbol=INTUCH
12,EPG,2022,3,"Fri, 10 Feb 2023 17:51:06 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832190&symbol=EPG


In [17]:
#df_out = df_out.drop(df_out.index[df_out['name'] == "SCC"])
df_out.shape[0]

18

In [18]:
sql = '''
SELECT *
FROM tickers
ORDER BY name'''
df_tickers = pd.read_sql(sql, conlt)
df_tickers.shape

(401, 9)

In [19]:
df_merge2 = pd.merge(df_out, df_tickers, on='name', how='outer', indicator=True)
df_merge2.shape

(405, 14)

### There are no ticker records

In [20]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link']
]

,name,year,quarter,published,link
5,NOK,2022.0,2.0,"Fri, 10 Feb 2023 18:18:08 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861834510&symbol=NOK
6,TDEX,2022.0,4.0,"Fri, 10 Feb 2023 18:02:48 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832900&symbol=TDEX
7,METCO,2022.0,1.0,"Fri, 10 Feb 2023 17:53:07 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832470&symbol=METCO
16,AFC,2022.0,2.0,"Fri, 10 Feb 2023 12:52:21 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861810060&symbol=AFC


In [21]:
df_merge2.loc[
    df_merge2['_merge'] == 'left_only',
    ['name','year','quarter','published','link','id','market']
].shape

(4, 7)

### There are ticker records

In [22]:
df_out2 = df_merge2.loc[
    df_merge2['_merge'] == 'both',
    ['name','year','quarter','published','link','id','market']
]
df_out2

,name,year,quarter,published,link,id,market
0,KTC,2022.0,4.0,"Fri, 10 Feb 2023 19:33:35 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861840250&symbol=KTC,259.0,SET50 / SETTHSI
1,GGC,2022.0,4.0,"Fri, 10 Feb 2023 19:07:36 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861839290&symbol=GGC,188.0,sSET
2,SCCC,2022.0,4.0,"Fri, 10 Feb 2023 18:56:28 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861838420&symbol=SCCC,428.0,SETCLMV / SETTHSI
3,TPIPP,2022.0,4.0,"Fri, 10 Feb 2023 18:39:46 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836440&symbol=TPIPP,560.0,SET
4,GPSC,2022.0,4.0,"Fri, 10 Feb 2023 18:35:41 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836180&symbol=GPSC,197.0,SET50 / SETTHSI
8,INTUCH,2022.0,4.0,"Fri, 10 Feb 2023 17:53:03 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832440&symbol=INTUCH,224.0,SET50 / SETCLMV / SETHD / SETTHSI
9,EPG,2022.0,3.0,"Fri, 10 Feb 2023 17:51:06 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832190&symbol=EPG,162.0,SET100 / SETTHSI
10,TOP,2022.0,4.0,"Fri, 10 Feb 2023 17:49:21 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832030&symbol=TOP,555.0,SET50 / SETCLMV / SETTHSI
11,INOX,2022.0,4.0,"Fri, 10 Feb 2023 17:21:43 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861829110&symbol=INOX,674.0,SET
12,KYE,2022.0,3.0,"Fri, 10 Feb 2023 17:16:58 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861828700&symbol=KYE,262.0,SET


In [23]:
df_out2 = df_out2[df_out2.year.notnull()]
df_out2.shape

(14, 7)

In [24]:
df_out2['year'] = df_out2['year'].astype(int)
df_out2['quarter'] = df_out2['quarter'].astype(int)
df_out2.shape

(14, 7)

In [26]:
file_name = 'F45-CLEAN-' + mmdd_str + '.csv'
clean_file = clean_path + file_name
output_file = csv_path + file_name
one_file = one_path + file_name

df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(output_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(clean_file, header=True, index=False, sep=',')
df_out2[['name','year','quarter','published','market']].sort_values(['published'],ascending=[False]).to_csv(one_file, header=True, index=False, sep=',')

In [27]:
sql = '''
SELECT * 
FROM epss
WHERE year = 2022'''
df_epss = pd.read_sql(sql, conlt)
df_epss.shape

(709, 14)

In [28]:
df_merge3 = pd.merge(df_out2, df_epss, on=['name','year','quarter'], how='outer', indicator=True)
df_merge3.shape

(719, 19)

### Already input, display profit amt & eps to check with new F45

In [29]:
df_merge3[df_merge3['_merge'] == 'both']

,name,year,quarter,published,link,id_x,market,id_y,q_amt,y_amt,aq_amt,ay_amt,q_eps,y_eps,aq_eps,ay_eps,ticker_id,publish_date,_merge
0,KTC,2022,4,"Fri, 10 Feb 2023 19:33:35 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861840250&symbol=KTC,259.0,SET50 / SETTHSI,22171.0,1665855.0,1247982.0,7079399.0,5878693.0,0.65,0.48,2.75,2.28,259.0,2023-01-20,both
10,PTTEP,2022,4,"Fri, 10 Feb 2023 17:12:50 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861827920&symbol=PTTEP,384.0,SET50 / SETCLMV / SETHD / SETTHSI,22179.0,15610584.0,10645266.0,70901335.0,38863595.0,3.93,2.67,17.94,9.70,384.0,2023-01-30,both
12,AOT,2022,1,"Fri, 10 Feb 2023 17:01:34 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861825720&symbol=AOT,24.0,SET50 / SETTHSI / SETWB,21109.0,-4271662.0,-3441979.0,-4271662.0,-3441979.0,-0.30,-0.24,-0.30,-0.24,24.0,2022-02-10,both
13,MC,2022,2,"Fri, 10 Feb 2023 08:08:24 +0700",https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861803980&symbol=MC,291.0,sSET,21892.0,230923.0,226463.0,254520.0,335188.0,0.29,0.29,0.32,0.42,291.0,2022-02-11,both


In [30]:
df_merge3[df_merge3['_merge'] == 'both'].shape

(4, 19)

### New F-45

In [31]:
df_inp2 = df_merge3[df_merge3['_merge'] == 'left_only']
df_inp3 = df_inp2.copy()
df_inp3.shape

(10, 19)

In [32]:
df_inp3['year'] = df_inp3['year'].astype(str)
df_inp3['quarter'] = df_inp3['quarter'].astype(str)
final = df_inp3.sort_values('name',ascending=True)
final_str = final.name+' '+final.year+' '+final.quarter+' '+final.link
final_str

6           EPG 2022 3 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832190&symbol=EPG
1           GGC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861839290&symbol=GGC
4         GPSC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836180&symbol=GPSC
8         INOX 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861829110&symbol=INOX
5     INTUCH 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832440&symbol=INTUCH
9           KYE 2022 3 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861828700&symbol=KYE
11          PSL 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861826120&symbol=PSL
2         SCCC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861838420&symbol=SCCC
7           TOP 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832030&symbol=TOP
3

In [33]:
df_inp3_q3 = df_inp3[df_inp3['quarter'] == '4']
final_q3 = df_inp3_q3.sort_values('name',ascending=True)
final_str_q3 = final_q3.name+' '+final_q3.year+' '+final_q3.quarter+' '+final_q3.link
final_str_q3

1           GGC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861839290&symbol=GGC
4         GPSC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836180&symbol=GPSC
8         INOX 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861829110&symbol=INOX
5     INTUCH 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832440&symbol=INTUCH
11          PSL 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861826120&symbol=PSL
2         SCCC 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861838420&symbol=SCCC
7           TOP 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832030&symbol=TOP
3       TPIPP 2022 4 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861836440&symbol=TPIPP
dtype: object

In [34]:
df_inp3_q1 = df_inp3[df_inp3['quarter'] != '4']
final_q1 = df_inp3_q1.sort_values('name',ascending=True)
final_str_q1 = final_q1.name+' '+final_q1.year+' '+final_q1.quarter+' '+final_q1.link
final_str_q1

6    EPG 2022 3 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861832190&symbol=EPG
9    KYE 2022 3 https://www.set.or.th/en/market/news-and-alert/newsdetails?id=16759861828700&symbol=KYE
dtype: object